# Exp 026 — Reward-model response reranker (Blind-A)

**Generation-side single-axis change vs 021 baseline.**

**Prerequisite**: `colab/Train_Reward_Model.ipynb` has completed and the resulting `reward_model.zip` is in `/content/drive/MyDrive/recsys2026-reward-model/`. This notebook will download the reward model from Drive before inference.

**Mechanism**: for each Blind-A query
1. wRRF retrieves top-20 candidates (same as 021).
2. Qwen 2.5-1.5B generates K=3 responses via temperature sampling (T=0.3, 0.7, 1.0).
3. Reward model (MiniLM-L-6 cross-encoder fine-tuned on train goal_progress_assessments) scores each (context, response) pair.
4. Ship the highest-scored response per query.

**Retrieval unchanged** (wRRF). nDCG@20 ≈ 0.19 expected.

**Target**: LLM judge 3.15 → ~3.3–3.5 if reward-model AUC > 0.70. Composite 0.33 → ~0.34–0.36.

Wall time: ~6–8 min on A100 (3× LM generation cost + reward scoring). Output → `/content/prediction.zip` + Drive backup.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone fresh-model from GitHub.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 3) Install deps (+ sentence-transformers for the reward model).
!pip install -q -r requirements.txt sentence-transformers
!python -c "import torch, transformers, sentence_transformers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'st', sentence_transformers.__version__)"

In [ ]:
# 4) Pull the trained reward model from Drive -> models/reward_model/.
# Requires prior Train_Reward_Model.ipynb run that saved reward_model.zip
# to /content/drive/MyDrive/recsys2026-reward-model/.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
SRC_ZIP = '/content/drive/MyDrive/recsys2026-reward-model/reward_model.zip'
assert os.path.isfile(SRC_ZIP), (
    f'reward_model.zip not found at {SRC_ZIP}. Run Train_Reward_Model.ipynb first.'
)
os.makedirs('models/reward_model', exist_ok=True)
with zipfile.ZipFile(SRC_ZIP) as zf:
    zf.extractall('models/reward_model')
!ls -lh models/reward_model/ | head -10

In [ ]:
# 5) Experiment parameters.
TID = '026-reward-rerank-qwen15b-blindsetA'
BATCH_SIZE = 16  # 3 samples per query × batch_size = 48 concurrent generations; keep modest.
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 6) Run Blind-A inference. The reward-model response reranker activates
# automatically because the yaml sets response_reranker_type=reward_model.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 7) Validate + package CodaBench-compliant prediction.zip.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC), f'inference output missing at {SRC}'
with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)}')
assert len(rows) == 80
sample = rows[0]
required = {'session_id','user_id','turn_number','predicted_track_ids','predicted_response'}
assert not (required - set(sample.keys()))
assert len(sample['predicted_track_ids']) == 20
assert sample['predicted_response'].strip()

wl = sorted(len(r['predicted_response'].split()) for r in rows)
print(f'response words: p25={wl[20]} median={wl[40]} p75={wl[60]} max={wl[-1]}')
print(f'sample response[0]: {sample["predicted_response"][:300]!r}')

stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip
print('\nprediction.zip ready — CodaBench-compliant.')

In [ ]:
# 8a) Browser download.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 8b) Drive backup.
import os, shutil
dst = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/prediction.zip', f'{dst}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst)
!ls -lh {dst}

## After scoring, ping Claude with the 4 numbers

Expected attribution:
- **nDCG@20 should ≈ 0.19** (retrieval unchanged). If materially different, there's a pipeline issue.
- **LLM judge lift** is the whole point of this experiment. Target ≥3.3.
- **LexDiv** likely small lift (sampling across temps adds variety).

If this lifts cleanly, Path 2 (GRPO training on the same reward model) becomes the natural follow-up — same reward signal, policy update instead of sampling pick.